# Lab 03: Developing a Tiny Transformer Language Model

In [16]:
#install the library
!pip install -q datasets
!pip install torch


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/126.7 MB 7.5 MB/s eta 0:00:17
    --------------------------------------- 2.1/126.7 MB 6.3 MB/s eta 0:00:20
    --------------------------------------- 3.1/126.7 MB 6.5 MB/s eta 0:00:19
   - -------------------------------------- 4.5/126.7 MB 5.8 MB/s eta 0:00:22
   - -------------------------------------- 5.2/126.7 MB 5.3 MB/s eta 0:00:23
   - -------------------------------------- 6.0/126.7 MB 4.9 MB/s eta 0:00:25
   -- ------------------------------------- 6.8/126.7 MB 4.6 MB/


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#load the dataset
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories")
split="train[:20000]"

C:\Users\oshan\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [5]:
#display one story
print(dataset["train"][0]["text"])

story_length = len(dataset["train"][0]["text"])
print("Length of one story:", story_length)

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
Length of one story: 701


In [6]:
#display three story
for i in range(3):
    print(dataset["train"][i]["text"])
    print()
    

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. 

In [7]:
#Preprocessing and Tokenization
text = dataset["train"][0]["text"] # get the story text
print(text[:500])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [8]:
characters = sorted(list(set(text)))
vocab_size = len(characters)
print("Vocabulary size:", vocab_size)
print(characters[:50])

Vocabulary size: 39
['\n', ' ', '"', "'", ',', '.', '?', 'A', 'C', 'H', 'I', 'L', 'M', 'O', 'S', 'T', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'x', 'y']


In [9]:
# convert in to numbers 
char_to_int = {char: i for i, char in enumerate(characters)}
print(char_to_int)

{'\n': 0, ' ': 1, '"': 2, "'": 3, ',': 4, '.': 5, '?': 6, 'A': 7, 'C': 8, 'H': 9, 'I': 10, 'L': 11, 'M': 12, 'O': 13, 'S': 14, 'T': 15, 'Y': 16, 'a': 17, 'b': 18, 'c': 19, 'd': 20, 'e': 21, 'f': 22, 'g': 23, 'h': 24, 'i': 25, 'k': 26, 'l': 27, 'm': 28, 'n': 29, 'o': 30, 'p': 31, 'r': 32, 's': 33, 't': 34, 'u': 35, 'w': 36, 'x': 37, 'y': 38}


In [10]:
#converts the story into numbers
encoded_text = [char_to_int[char] for char in text]
print("First 20 encoded values:", encoded_text[:20])

First 20 encoded values: [13, 29, 21, 1, 20, 17, 38, 4, 1, 17, 1, 27, 25, 34, 34, 27, 21, 1, 23, 25]


In [11]:
# input and target sequences
input_text = encoded_text[:-1]
target_text = encoded_text[1:]

print("First 20 input values:", input_text[:20])
print("First 20 target values:", target_text[:20])


First 20 input values: [13, 29, 21, 1, 20, 17, 38, 4, 1, 17, 1, 27, 25, 34, 34, 27, 21, 1, 23, 25]
First 20 target values: [29, 21, 1, 20, 17, 38, 4, 1, 17, 1, 27, 25, 34, 34, 27, 21, 1, 23, 25, 32]


In [13]:
#Prepare training data

selected_stories = [] # select small group of stories for training

for i in range(200):
    selected_stories.append(dataset["train"][i]["text"])

corpus = "\n".join(selected_stories)

print("Number of stories:", len(selected_stories))
print("Total characters:", len(corpus))
print(corpus[:500])


Number of stories: 200
Total characters: 159847
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b


In [14]:

# create character vocabulary 
characters = sorted(list(set(corpus)))
vocab_size = len(characters)

char_to_int = {char: i for i, char in enumerate(characters)}  #convert text into numbers
int_to_char = {i: char for i, char in enumerate(characters)}

encoded_text = [char_to_int[char] for char in corpus]

print("Vocabulary size:", vocab_size)
print("Total encoded characters:", len(encoded_text))
print("First 20 encoded values:", encoded_text[:20])


Vocabulary size: 70
Total encoded characters: 159847
First 20 encoded values: [27, 51, 42, 1, 41, 38, 62, 5, 1, 38, 1, 49, 46, 57, 57, 49, 42, 1, 44, 46]


In [17]:
import torch
print(torch.__version__)

2.15.0+cpu


In [18]:
#Split into training and validation data
data = torch.tensor(encoded_text, dtype=torch.long)
split_index = int(0.9 * len(data))

train_data = data[:split_index]
val_data = data[split_index:]

print("Training data length:", len(train_data))
print("Validation data length:", len(val_data))

Training data length: 143862
Validation data length: 15985


In [19]:

# set the model and training settings
device = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 64
batch_size = 32
learning_rate = 0.001

n_embd = 64
n_head = 4
n_layer = 2

print("Device:", device)


Device: cpu


In [20]:

# get input and target data
def get_batch(data):
    positions = torch.randint(len(data) - block_size - 1, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in positions])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in positions])

    return x.to(device), y.to(device)

xb, yb = get_batch(train_data)

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)


Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])


In [42]:
# create the Transformer model
import torch
import torch.nn as nn

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=n_embd,
            nhead=n_head,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=n_layer
        )

        self.output_layer = nn.Linear(
            n_embd, vocab_size
        )

    def forward(self, idx):
        x = self.token_embedding(idx)

        # add position information
        positions = torch.arange(idx.shape[1], device=idx.device)
        x = x + self.position_embedding(positions)

        # prevent it from looking at future characters
        mask = torch.triu(
            torch.ones(idx.shape[1], idx.shape[1], device=idx.device),
            diagonal=1
        ).bool()

        x = self.transformer(x, mask=mask)

        # predict the next character
        logits = self.output_layer(x)

        return logits

In [33]:
# set up the loss function and optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [34]:
#train the model
print("Model is ready:", model is not None)
print("Training data shape:", train_data.shape)
print("Validation data shape:", val_data.shape)

Model is ready: True
Training data shape: torch.Size([143862])
Validation data shape: torch.Size([15985])


In [50]:
#initialize the model
model = TinyTransformer(
    vocab_size,
    n_embd,
    n_head,
    n_layer,
    block_size
).to(device)

print("Model initialized:", model is not None)
print("Model device:", next(model.parameters()).device)

Model initialized: True
Model device: cpu


In [48]:
print(type(model))
print("forward" in type(model).__dict__)

<class '__main__.TinyTransformer'>
True


In [ ]:
#test the model
xb, yb = get_batch(train_data)

logits = model(xb)

loss = loss_function(
    logits.view(-1, vocab_size),
    yb.view(-1)
)

print("Input shape:", xb.shape)
print("Output shape:", logits.shape)
print("Loss:", loss.item())

Input shape: torch.Size([32, 64])
Output shape: torch.Size([32, 64, 70])
Loss: 4.43220329284668


In [53]:
#Train the Model
max_iters = 500
eval_interval = 100

# prepare to train the model
for step in range(max_iters):
    xb, yb = get_batch(train_data)

    logits = model(xb)

    loss = loss_function(
        logits.view(-1, vocab_size),
        yb.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        print("Step:", step, "Loss:", loss.item())

Step: 0 Loss: 4.422945976257324
Step: 100 Loss: 2.4570953845977783
Step: 200 Loss: 2.2478508949279785
Step: 300 Loss: 2.180328845977783
Step: 400 Loss: 2.018156051635742


In [52]:
# connect the optimizer to the current model
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

In [55]:
#monitor training
# check the loss on training and validation data
@torch.no_grad()
def estimate_loss():
    model.eval()

    xb, yb = get_batch(train_data)
    train_loss = loss_function(model(xb).view(-1, vocab_size), yb.view(-1)).item()

    xb, yb = get_batch(val_data)
    val_loss = loss_function(model(xb).view(-1, vocab_size), yb.view(-1)).item()

    model.train()

    return train_loss, val_loss

In [56]:
#check the loss
train_loss, val_loss = estimate_loss()

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 1.8818607330322266
Validation loss: 2.0064120292663574


In [57]:
# evaluate the model
import math
perplexity = math.exp(val_loss)

print("Validation loss:", val_loss)
print("Perplexity:", perplexity)

Validation loss: 2.0064120292663574
Perplexity: 7.4365871653324485


In [70]:
print("Model vocabulary:", model.token_embedding.num_embeddings)

print("Current vocabulary:", len(stoi))

print("Available text and vocabulary variables:")
print([
    name for name in globals()
    if any(word in name.lower()
           for word in ["text", "vocab", "char", "stoi", "itos", "dataset"])
])

Model vocabulary: 70
Current vocabulary: 39
Available text and vocabulary variables:
['load_dataset', 'dataset', 'text', 'characters', 'vocab_size', 'char_to_int', 'encoded_text', 'input_text', 'target_text', 'int_to_char', 'context', 'next_char', 'context_crop', 'next_character', 'chars', 'stoi', 'itos']


In [75]:
# Generate a story

# Use the vocabulary created during preprocessing
int_to_char = {i: ch for ch, i in char_to_int.items()}

print("Vocabulary size:", len(char_to_int))
print("Model vocabulary size:", model.token_embedding.num_embeddings)

prompt = "Once upon a time"

# Convert the prompt into numbers
prompt_ids = [char_to_int[ch] for ch in prompt]

context = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)

model.eval()

with torch.no_grad():
    for _ in range(500):
        context_crop = context[:, -block_size:]

        logits = model(context_crop)

        temperature = 0.5
        probabilities = torch.softmax(logits[0, -1] / temperature, dim=0)
        next_char = torch.multinomial(probabilities, num_samples=1)

        context = torch.cat(
            (context, next_char.view(1, 1)),
            dim=1
        )

model.train()

# Convert the generated numbers back into text
generated_text = "".join(
    int_to_char[int(number)] for number in context[0].tolist()
)

print(generated_text)

Vocabulary size: 70
Model vocabulary size: 70
Once upon a time the was a big was the was she stried nooked ther and the was he it was a be was the she she she hourl ound and tohe the gir plaring and the bigh ling wase ands she the like. The lime was hey she be bet was a she to care to bo the plar and oun.

The round she wer nown. 

The was was the you was the rall a smand the gre she was and st was a fastay the pooked st the she wise so was hat happy a shappy the was and the she bo the was and se she wase shey a the mind he was a boy said to the she gas an


### Experiment with Model Settings
 **Experiment 1: Generated Text Length**

The number of generated characters was increased from 300 to 500. The model generated a longer story, but the quality of the text depended on how well the model had learned from the training data.

**Experiment 2: Temperature**

The temperature was set to 0.5. A lower temperature makes the model choose more predictable characters, which may make the generated text less random. However, the story may still contain errors because the model was trained on a small dataset.

In [76]:
torch.save(model.state_dict(), "tiny_transformer.pth")

In [77]:
model_config = {                       #Save the model configuration
    "vocab_size": vocab_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "block_size": block_size
}

import json

with open("tiny_transformer_config.json", "w") as file:
    json.dump(model_config, file, indent=4)